In [1]:
import numpy as np
import h5py
import os
import pandas as pd
import kipoiseq

In [8]:
DIR = "/eagle/AIHPC4Edu/ssalazar/projects/enformer_training/full_393216bp"
val_seqs = os.path.join(DIR, "human_validation.h5")
DIR_val_freqs = "/eagle/AIHPC4Edu/ssalazar/projects/enformer_training/EUR_allele_freqs_val"

In [4]:
with h5py.File(val_seqs, 'r') as h5f:
    print(h5f.keys())

<KeysViewHDF5 ['query_region', 'sequence', 'target']>


In [5]:
base_to_index = {'A': 0, 'C': 1, 'G': 2, 'T': 3}


In [6]:
def get_popseq(interval, ref_seq, DIR_freqs, length):
    '''
    start: kipoi expanded interval
    ref_seq: one-hot-encoded reference sequence
    DIR_freqs: directory path where allele dosages per chromosome are stored
    '''
    coordinates = np.arange(interval.start + 1, interval.start + 1 + length) # 393_216 long vector of genomic coordinates

    dosages = pd.read_csv(os.path.join(DIR_freqs, f'{interval.chrom}_EUR_allele_freqs.txt'), sep=' ', header=None)
    dosage_series = pd.Series(dosages[3].values, index=dosages[1])
    aligned_dosages = dosage_series.reindex(coordinates, fill_value=0) # vector of zeroes and alternative allele dosages of length 393_216
    
    dosages[2] = dosages[2].map(base_to_index) # map dosage values to second dimension depending on base
    allele_series = pd.Series(dosages[2].values, index=dosages[1])
    aligned_alleles = allele_series.reindex(coordinates, fill_value=-1).values # vector of -1s and alternate allele position as index of second dimension

    mask = aligned_alleles != -1
    rows = np.where(mask)[0] # which nucleotide positions need updating [10, 22] (positions)
    alt_cols = aligned_alleles[mask] # the column index of the alternate allele [2, 3] G -> 2, T -> 3
    alt_vals = aligned_dosages[mask] # the alterante allele dosage [0.2, 0.6] 
    ref_cols = ref_seq[rows].argmax(axis=1) # get the column index where the reference base was

    result = ref_seq
    result[rows, ref_cols] = 1.0 - alt_vals # reduce reference base probability
    result[rows, alt_cols] = alt_vals  # assign alt base probability
    # result = np.zeros((length, 4), dtype=aligned_dosages.dtype)
    # rows = np.arange(length)
    # cols = aligned_alleles
    # result[rows, cols] = aligned_dosages # zeroes and allele dosages values in correct position
    # mask = (result != 0).any(axis=1)
    # ref_seq[mask] = result[mask] # replace the non-zero allele dosages on reference one hot encoded sequence
    return result

In [9]:
with h5py.File(val_seqs, 'r+') as h5f:
    num_seqs = h5f['sequence'].shape[0]

    # Create dataset if it doesn't exist yet
    if 'pop_sequence' not in h5f:
        h5f.create_dataset(
            'pop_sequence',
            shape=(0, 393216, 4),
            maxshape=(None, 393216, 4),
            dtype=np.float32
        )

    for n_seq in range(num_seqs):
        if n_seq % 30 == 0: 
            print(f"{n_seq+1}/{(num_seqs)}")
        chr, start, end = h5f['query_region'][n_seq, :]
        if chr == 0:
            chr = 'X'
        else:
            chr = int(chr)
        ref_seq = h5f['sequence'][n_seq, :]

        expanded_interval = kipoiseq.Interval(
            chrom=f'chr{chr}',
            start=int(start),
            end=int(end)
        ).resize(393216)

        pop_seq = get_popseq(expanded_interval, ref_seq=ref_seq, DIR_freqs=DIR_val_freqs, length=393216)

        # Resize the dataset to fit new entry
        h5f['pop_sequence'].resize((n_seq + 1, 393216, 4))

        # Write the pop_seq into the dataset
        h5f['pop_sequence'][n_seq, :, :] = pop_seq


1/2213
31/2213
61/2213
91/2213
121/2213
151/2213
181/2213
211/2213
241/2213
271/2213
301/2213
331/2213
361/2213
391/2213
421/2213
451/2213
481/2213
511/2213
541/2213
571/2213
601/2213
631/2213
661/2213
691/2213
721/2213
751/2213
781/2213
811/2213
841/2213
871/2213
901/2213
931/2213
961/2213
991/2213
1021/2213
1051/2213
1081/2213
1111/2213
1141/2213
1171/2213
1201/2213
1231/2213
1261/2213
1291/2213
1321/2213
1351/2213
1381/2213
1411/2213
1441/2213
1471/2213
1501/2213
1531/2213
1561/2213
1591/2213
1621/2213
1651/2213
1681/2213
1711/2213
1741/2213
1771/2213
1801/2213
1831/2213
1861/2213
1891/2213
1921/2213
1951/2213
1981/2213
2011/2213
2041/2213
2071/2213
2101/2213
2131/2213
2161/2213
2191/2213


In [10]:
with h5py.File(val_seqs, 'r') as h5f:
    region = h5f['query_region'][231,:]
    popseq = h5f['pop_sequence'][231,:]
    refseq = h5f['sequence'][231,:]

In [12]:
int(region[2])

230419611

232:chr2_230288539_230419611

chr2	230157468	230550684 expanded

In [13]:
import kipoiseq

In [14]:
chr = f'chr{int(region[0])}'
start = int(region[1])
end = int(region[2])
expanded_interval = kipoiseq.Interval(
            chrom=chr,
            start=int(start),
            end=int(end)
        ).resize(393_216)

In [15]:
expanded_interval 

Interval(chrom='chr2', start=230157467, end=230550683, name='', strand='.', ...)

In [16]:
coordinates = np.arange(expanded_interval.start + 1, expanded_interval.start + 1 + 393_216) # 393_216 long vector of genomic coordinates

In [17]:
coordinates[13400:13411]

array([230170868, 230170869, 230170870, 230170871, 230170872, 230170873,
       230170874, 230170875, 230170876, 230170877, 230170878])

In [18]:
popseq[13400:13411,:]

array([[1.      , 0.      , 0.      , 0.      ],
       [0.      , 1.      , 0.      , 0.      ],
       [0.      , 1.      , 0.      , 0.      ],
       [1.      , 0.      , 0.      , 0.      ],
       [0.      , 0.      , 0.      , 1.      ],
       [0.      , 0.108527, 0.      , 0.891473],
       [0.      , 0.      , 0.      , 1.      ],
       [0.      , 0.      , 1.      , 0.      ],
       [1.      , 0.      , 0.      , 0.      ],
       [0.      , 1.      , 0.      , 0.      ],
       [0.      , 1.      , 0.      , 0.      ]], dtype=float32)

In [ ]:
refseq[13400:13411,:]

array([[1., 0., 0., 0.],
       [0., 1., 0., 0.],
       [0., 1., 0., 0.],
       [1., 0., 0., 0.],
       [0., 0., 0., 1.],
       [0., 0., 0., 1.],
       [0., 0., 0., 1.],
       [0., 0., 1., 0.],
       [1., 0., 0., 0.],
       [0., 1., 0., 0.],
       [0., 1., 0., 0.]], dtype=float32)

chr2 230170873 C 0.108527